# Problem 2: Phi-2 with SynCode, then Python arithmetic

`run_syncode(textual_query)` returns `([left_operand, right_operand], operator_string, answer_float)`.
[Phi-2](https://huggingface.co/microsoft/phi-2) extracts the operands and the operator under a SynCode
grammar; Python computes the answer. The code follows SynCode's
[logits-processor example](https://github.com/structuredllm/syncode/blob/main/notebooks/example_logits_processor.ipynb).

In [1]:
%pip -q install "syncode==0.4.16" "transformers==4.53.2" "accelerate==1.15.0" "lark==1.3.1"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import operator

import torch
from syncode import Grammar, SyncodeLogitsProcessor
from transformers import AutoModelForCausalLM, AutoTokenizer

device = 'cuda'
model_name = "microsoft/phi-2"
# FP32: Phi-2's attention can overflow in FP16.
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32).eval().to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)

/srv/condor/execute/dir_1355557/vlane_cs521hw3-p2v4-20261027/venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Fetching 2 files:  50%|█████     | 1/2 [00:05<00:05,  5.25s/it]

Fetching 2 files: 100%|██████████| 2/2 [00:09<00:00,  4.80s/it]

Fetching 2 files: 100%|██████████| 2/2 [00:09<00:00,  4.80s/it]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:  50%|█████     | 1/2 [00:00<00:00,  1.36it/s]

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.47it/s]

In [3]:
# One lexical terminal keeps valid decimal and multi-character token prefixes
# available in SynCode 0.4.16 (verified token-by-token with the Phi-2 tokenizer).
grammar_str = r'''
start: EXTRACTION
EXTRACTION: /\{"operands": \[-?[0-9]+(\.[0-9]+)?([eE][+-]?[0-9]+)?, -?[0-9]+(\.[0-9]+)?([eE][+-]?[0-9]+)?\], "operator": "[+*\/-]"\}/
%import common.WS
%ignore WS
'''
syncode_logits_processor = SyncodeLogitsProcessor(grammar=Grammar(grammar_str), tokenizer=tokenizer, parse_output_only=True)

[2026-10-27 22:39:48,287-syncode.mask_store.mask_store] - Using cache: True and fsm path cache/mask_stores/CodeGenTokenizerFast/grammar_mask_6368143039_50257.pkl exist: False


[2026-10-27 22:39:48,288-syncode.mask_store.mask_store] - Creating mask store for CodeGenTokenizerFast and custom, may take more than 10 minutes. Caching at /srv/condor/execute/dir_1355557/vlane_cs521hw3-p2v4-20261027/work/cache/mask_stores/CodeGenTokenizerFast/grammar_mask_6368143039_50257.pkl.


[2026-10-27 22:39:48,731-syncode.mask_store.fsm_set] - 2 FSMs with 59 states initialized in 0.01 seconds


[2026-10-27 22:39:48,732-syncode.mask_store.mask_store] - Ignore whitespace is True


[2026-10-27 22:39:48,733-syncode.mask_store.mask_store] - Number of 2 length terminal sequences reduced from 4 to 2


...

 97%|█████████▋| 57/59 [00:06<00:00,  9.12it/s]

 98%|█████████▊| 58/59 [00:06<00:00,  9.12it/s]

100%|██████████| 59/59 [00:06<00:00,  7.81it/s]

100%|██████████| 59/59 [00:06<00:00,  8.51it/s]

[2026-10-27 22:39:55,679-syncode.mask_store.mask_store] - Time taken to create mask store: 7.37 seconds


In [4]:
FEW_SHOT_PROMPT = '''Extract the two numbers and the arithmetic operator from the query as JSON. Do not calculate the answer.
Copy complete numbers, including minus signs, decimal digits, and scientific-notation exponents.
Keep the operands in their original order, including for addition and multiplication.
Exception: "subtract A from B" requires operands [B, A].

Query: What is 2.1e2 multiplied by 1.2?
JSON: {{"operands": [2.1e2, 1.2], "operator": "*"}}

Query: What is -4.3 plus 7.28?
JSON: {{"operands": [-4.3, 7.28], "operator": "+"}}

Query: What is 120.4 divided by 4.0?
JSON: {{"operands": [120.4, 4.0], "operator": "/"}}

Query: Subtract 6.95 from -18.8.
JSON: {{"operands": [-18.8, 6.95], "operator": "-"}}

Query: {query}
JSON: '''

OPERATORS = {'+': operator.add, '-': operator.sub, '*': operator.mul, '/': operator.truediv}


def run_syncode(textual_query):
    """Return (ordered list of operands, operator string, answer float)."""
    prompt = FEW_SHOT_PROMPT.format(query=textual_query.strip())
    syncode_logits_processor.reset()
    inputs = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
    output = model.generate(inputs, max_new_tokens=80, pad_token_id=tokenizer.eos_token_id,
                            logits_processor=[syncode_logits_processor])
    extraction = json.loads(tokenizer.decode(output[0][len(inputs[0]):], skip_special_tokens=True))
    a, b = (float(v) for v in extraction["operands"])
    op = extraction["operator"]
    return [a, b], op, OPERATORS[op](a, b)

## Required examples

In [5]:
for query in ["What is 327. multiplied by 11.0?", "What is 45.1 plus 23.54?", "What is 120.4 divided by 4.0?"]:
    print(query, run_syncode(query))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


What is 327. multiplied by 11.0? ([327.0, 11.0], '*', 3597.0)


What is 45.1 plus 23.54? ([45.1, 23.54], '+', 68.64)


What is 120.4 divided by 4.0? ([120.4, 4.0], '/', 30.1)
